In [1]:
# mike babb
# created: 2026 08 23
# udpated: 2026 09 12
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase


In [3]:
import networkx as nx
import numpy as np
import pandas as pd

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA WORD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars,word_byte,word_id
0,abdom,abdom,5,abdmo,"{m, b, a, o, d}",5,20491,0
1,abend,abend,5,abden,"{e, b, n, a, d}",5,8219,1
2,abets,abets,5,abest,"{e, b, s, t, a}",5,786451,2
3,abhor,abhor,5,abhor,"{r, b, h, a, o}",5,147587,3
4,abide,abide,5,abdei,"{e, b, a, i, d}",5,283,4


# IMPORT DATA

In [7]:
l5_df = pd.read_csv(filepath_or_buffer='l5.txt', sep = '\t', dtype = np.int32)

In [8]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327


# ASSIGN WORDS TO IDS

In [9]:
# how many unique letter groups?
for il in range(2, 6):
    cn  = f"l{il}"
    print(cn, l5_df[cn].unique().shape)

l2 (2005,)
l3 (2131,)
l4 (649,)
l5 (11,)


In [10]:
for idx in range(1, 6):
    cn = f"w{str(idx)}b"
    ncn = f"w{str(idx)}"
    l5_df[ncn] = l5_df[cn].map(word_byte_to_word_dict)

In [11]:
# group the words, and sort
output_list = []

for i_row, row in l5_df.iterrows():
    my_set = set()
    l5_letter_set = set()
    for cn_idx in range(1, 6):
        cn = f"w{cn_idx}"        
        my_word = row[cn]
        my_set.add(my_word)
        
    output = tuple(sorted(my_set))
    
    output_list.append(output)
    

In [12]:
l5_df['word_group'] = output_list

In [13]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5,w1,w2,w3,w4,w5,word_group
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327,ambry,fldxt,pucks,vejoz,whing,"(ambry, fldxt, pucks, vejoz, whing)"
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327,ambry,fldxt,pucks,whing,vejoz,"(ambry, fldxt, pucks, vejoz, whing)"
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327,ambry,fldxt,pungs,vejoz,whick,"(ambry, fldxt, pungs, vejoz, whick)"
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327,ambry,fldxt,pungs,whick,vejoz,"(ambry, fldxt, pungs, vejoz, whick)"
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327,ambry,fldxt,vejoz,pucks,whing,"(ambry, fldxt, pucks, vejoz, whing)"


In [14]:
# looking at the first two rows, we see identical word groups
# let's drop duplicated word_groups


In [15]:
l5_df = l5_df.drop_duplicates(subset='word_group').reset_index(drop = True)

In [16]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5,w1,w2,w3,w4,w5,word_group
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327,ambry,fldxt,pucks,vejoz,whing,"(ambry, fldxt, pucks, vejoz, whing)"
1,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327,ambry,fldxt,pungs,vejoz,whick,"(ambry, fldxt, pungs, vejoz, whick)"
2,25202689,850,2121764,458888,35130368,25203539,27325303,27784191,62914559,ampyx,bejig,fconv,hdqrs,klutz,"(ampyx, bejig, fconv, hdqrs, klutz)"
3,25202689,4194642,2121764,458888,35130368,29397331,31519095,31977983,67108351,ampyx,bewig,fconv,hdqrs,klutz,"(ampyx, bewig, fconv, hdqrs, klutz)"
4,25202689,34226178,6291844,2616,1320000,59428867,65720711,65723327,67043327,ampyx,bortz,chivw,fjeld,gunks,"(ampyx, bortz, chivw, fjeld, gunks)"


In [17]:
l5_df.shape

(538, 15)

In [18]:
# concatenate all letters together after sorting
l5_df['l5_letter_group'] = l5_df['word_group'].map(lambda x: ''.join(sorted(''.join(x))))

In [19]:
# identify the remaining letter not used
l5_df['l5_remainder_letter'] = l5_df['l5_letter_group'].map(lambda x: ''.join(set(ascii_lowercase).difference(x)))

In [20]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5,w1,w2,w3,w4,w5,word_group,l5_letter_group,l5_remainder_letter
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327,ambry,fldxt,pucks,vejoz,whing,"(ambry, fldxt, pucks, vejoz, whing)",abcdefghijklmnoprstuvwxyz,q
1,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327,ambry,fldxt,pungs,vejoz,whick,"(ambry, fldxt, pungs, vejoz, whick)",abcdefghijklmnoprstuvwxyz,q
2,25202689,850,2121764,458888,35130368,25203539,27325303,27784191,62914559,ampyx,bejig,fconv,hdqrs,klutz,"(ampyx, bejig, fconv, hdqrs, klutz)",abcdefghijklmnopqrstuvxyz,w
3,25202689,4194642,2121764,458888,35130368,29397331,31519095,31977983,67108351,ampyx,bewig,fconv,hdqrs,klutz,"(ampyx, bewig, fconv, hdqrs, klutz)",abcdefghiklmnopqrstuvwxyz,j
4,25202689,34226178,6291844,2616,1320000,59428867,65720711,65723327,67043327,ampyx,bortz,chivw,fjeld,gunks,"(ampyx, bortz, chivw, fjeld, gunks)",abcdefghijklmnoprstuvwxyz,q


In [21]:
l5_df = l5_df.sort_values(by = ['w1', 'w2', 'w3', 'w4', 'w5']).reset_index(drop = True)

In [22]:
# these are the different combinations
l5_df.shape

(538, 17)

In [23]:
# how many unique letters not used?
l5_df['l5_remainder_letter'].unique().shape


(11,)

In [24]:
# and how often are they used?
l5_df['l5_remainder_letter'].value_counts()

l5_remainder_letter
q    467
z     18
j     13
v      9
x      8
g      7
b      6
w      5
m      2
f      2
c      1
Name: count, dtype: int64

In [25]:
import collections

In [26]:
# how many words are used?
my_counter = collections.Counter()
for widx in range(1, 6):
    cn = f"w{widx}"
    my_counter.update(l5_df[cn].tolist())

In [27]:
len(my_counter)

493

# EXPORT EACH WORD GROUP BY REMAINDER LETTER TO AN EXCEL SHEET

In [28]:
ofn = 'l5_output.xlsx'
ofpn = os.path.join(rc.output_folder, ofn)
e_writer = pd.ExcelWriter(path = ofn)
col_names = ['w1', 'w2', 'w3', 'w4', 'w5', 'word_group', 'l5_remainder_letter']
rem_letters = sorted(l5_df['l5_remainder_letter'].unique().tolist())

for rl in rem_letters:
    tdf = l5_df.loc[l5_df['l5_remainder_letter'] == rl, col_names].sort_values(by=['word_group']).reset_index(drop = True)

    print(rl, tdf.shape)    

    tdf = tdf.drop(labels='word_group', axis = 1)
    sheet_name = f"remainder_{rl}"
    tdf.to_excel(excel_writer=e_writer, sheet_name=sheet_name, index = False)       

b (6, 7)
c (1, 7)
f (2, 7)
g (7, 7)
j (13, 7)
m (2, 7)
q (467, 7)
v (9, 7)
w (5, 7)
x (8, 7)
z (18, 7)


# COUNT THE OCCURENCE OF EACH WORD ACROSS WORD GROUPS BY REMAINDER LETTER

In [29]:
melt_col_names = ['w1', 'w2', 'w3', 'w4', 'w5', 'l5_remainder_letter']
mdf = pd.melt(frame = l5_df[melt_col_names], id_vars = ['l5_remainder_letter'], var_name = 'w_idx', value_name = 'word')

In [30]:
mdf['word_count'] = int(1)

In [31]:
mdf_pivot = pd.pivot_table(data = mdf, values = 'word_count', index = 'word',
                            columns = 'l5_remainder_letter',aggfunc='sum',
                            fill_value=0,margins=True, margins_name='Unique Word Groups')

In [32]:
mdf_pivot.shape

(494, 12)

In [33]:
# divide the last row by five in order to get the correct count of unique groups by remainder letter
# do this because each group has five words
mdf_pivot.iloc[-1] = (mdf_pivot.iloc[-1] / 5).astype(int)

In [34]:
mdf_pivot = mdf_pivot.reset_index(drop = False)

In [35]:
mdf_pivot.tail()

l5_remainder_letter,word,b,c,f,g,j,m,q,v,w,x,z,Unique Word Groups
489,zings,0,0,0,0,2,0,12,0,0,0,0,14
490,zingy,0,0,0,0,2,0,13,1,0,0,0,16
491,zygon,0,0,0,0,2,0,16,0,0,0,0,18
492,zymic,0,0,0,0,0,0,1,0,0,0,0,1
493,Unique Word Groups,6,1,2,7,13,2,467,9,5,8,18,538


In [36]:
# let's see how often words use more than one vowel
vowel_set = set('aeiouy')
mdf_pivot['vowels'] = mdf_pivot['word'].map(lambda x: ''.join(sorted(vowel_set.intersection(x))))
mdf_pivot['vowel_usage'] = mdf_pivot['vowels'].map(lambda x: len(x))

In [37]:
mdf_pivot.to_excel(excel_writer=e_writer, sheet_name='word_count',index = False)

In [38]:
# most common vowel usage
vowel_usage = mdf_pivot.iloc[:-1]['vowels'].value_counts().to_frame(name = 'word_count').reset_index(names = ['vowel_group'])
vowel_usage.loc[vowel_usage['vowel_group'] == '', 'vowel_group'] = '*no value used*'

In [39]:
vowel_usage.head()

,vowel_group,word_count
0,a,79
1,i,74
2,u,64
3,o,42
4,ay,39


In [40]:
vowel_usage.to_excel(excel_writer=e_writer, sheet_name='vowel_usage', index = False)

In [41]:
e_writer.close()